# Checkpoint 46 — Retention Policy Analysis

This notebook reviews the validation-selected, once-only test evaluation created by `src/optimize_retention_policy.py`. The policy is frozen before the 2025 test target is accessed.

In [1]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

processed = project_root / "data" / "processed"
comparison = pd.read_csv(processed / "retention_policy_comparison.csv")
decision = pd.read_csv(processed / "retention_policy_decision.csv")
model_metrics = pd.read_csv(processed / "retention_final_test_model_metrics.csv")
sensitivity = pd.read_csv(processed / "retention_policy_sensitivity.csv")
current_summary = pd.read_csv(processed / "current_retention_policy_summary.csv")
current_groups = pd.read_csv(processed / "current_retention_policy_group_summary.csv")
validation = pd.read_csv(processed / "retention_policy_validation.csv")

## 1. Policy comparison

The fixed 0.50 threshold, top-k rules, cost threshold, and budget-constrained policy are shown together. `Outcome-aligned net value` uses observed attrition outcomes but still assumes the intervention success rate; it is not observed profit.

In [2]:
comparison[[
    "evaluation_period",
    "policy",
    "selected_count",
    "selected_positive_cases",
    "precision",
    "capture_rate",
    "intervention_spend_usd",
    "expected_net_value_usd",
    "outcome_aligned_net_value_usd",
]]

,evaluation_period,policy,selected_count,selected_positive_cases,precision,capture_rate,intervention_spend_usd,expected_net_value_usd,outcome_aligned_net_value_usd
0,2024 validation,No intervention,0,0,0.000000,0.000000,0.0,0.000000e+00,0.0
1,2024 validation,Fixed 0.50 threshold,0,0,0.000000,0.000000,0.0,0.000000e+00,0.0
2,2024 validation,Validation cost-optimal threshold,2151,313,0.145514,0.534130,5377500.0,3.473619e+05,717575.0
3,2024 validation,Top 10% probability,553,109,0.197107,0.186007,1382500.0,2.363027e+05,514900.0
4,2024 validation,Top 700 probability,700,132,0.188571,0.225256,1750000.0,2.569048e+05,594400.0
5,2024 validation,Budget-constrained expected value,700,85,0.121429,0.145051,1750000.0,8.409739e+05,512125.0
6,2025 final test,No intervention,0,0,0.000000,0.000000,0.0,0.000000e+00,0.0
7,2025 final test,Fixed 0.50 threshold,0,0,0.000000,0.000000,0.0,0.000000e+00,0.0
8,2025 final test,Validation cost-optimal threshold,3294,500,0.151791,0.622665,8235000.0,1.805733e+06,2247675.0
9,2025 final test,Top 10% probability,665,132,0.198496,0.164384,1662500.0,8.947197e+05,779925.0


## 2. Frozen final decision

The SHA-256 value records the exact policy configuration that existed before the final test evaluation.

In [3]:
decision.T

,0
selected_policy,budget_expected_value
selected_policy_display_name,Budget-constrained expected value
validation_snapshot,2024-06-30
test_snapshot,2025-06-30
fixed_threshold,0.5
cost_optimal_threshold,0.106832
cost_optimal_validation_selected_count,2151
top_fraction,0.1
top_count,700
maximum_employees,700


## 3. Final model metrics

These metrics come from the previously reserved 2025 test snapshot. The policy must not be changed in response to them.

In [4]:
model_metrics

,evaluation_period,rows,positive_cases,positive_rate,mean_predicted_probability,mean_calibration_gap,brier_score,pr_auc,roc_auc,top_decile_count,top_decile_precision,top_decile_capture,top_decile_lift
0,2025 final test,6641,803,0.120916,0.110208,-0.010707,0.104848,0.171047,0.606113,665,0.198496,0.164384,1.641611


## 4. Cost sensitivity

The selected employee set is revalued across all 27 cost scenarios without changing the test policy.

In [5]:
selected_sensitivity = sensitivity.loc[
    sensitivity["policy_key"].eq("budget_expected_value")
    & sensitivity["evaluation_period"].eq("2025 final test")
]

selected_sensitivity[[
    "scenario_id",
    "expected_net_value_usd",
    "outcome_aligned_net_value_usd",
]].sort_values("outcome_aligned_net_value_usd")

,scenario_id,expected_net_value_usd,outcome_aligned_net_value_usd
83,low-intensive-conservative,-2.871902e+06,-2743260.0
191,base-intensive-conservative,-2.243805e+06,-1986520.0
95,low-intensive-base,-1.929756e+06,-1608150.0
299,high-intensive-conservative,-1.615707e+06,-1229780.0
47,low-standard-conservative,-1.121902e+06,-993260.0
107,low-intensive-optimistic,-9.876099e+05,-473040.0
155,base-standard-conservative,-4.938049e+05,-236520.0
11,low-lean-conservative,-7.190247e+04,56740.0
59,low-standard-base,-1.797562e+05,141850.0
203,base-intensive-base,-3.595124e+05,283700.0


## 5. Current synthetic plan

The current plan is for human review only. Financial value is not employee value, and the scores must not trigger automatic employment actions.

In [6]:
current_summary

,snapshot_date,eligible_employees,selected_for_human_review,intervention_budget_usd,projected_intervention_spend_usd,projected_expected_prevented_departures,projected_expected_avoided_cost_usd,projected_expected_net_value_usd,human_review_required,automatic_employment_action_permitted
0,2026-06-30,7305,700,1750000.0,1750000.0,35.034285,4.146216e+06,2.396216e+06,True,False


In [7]:
current_groups.sort_values("selection_rate", ascending=False)

,department_name,eligible_employees,selected_for_human_review,average_probability,average_expected_net_value_usd,selection_rate
4,Information Technology,734,152,0.127715,1099.591337,0.207084
1,Engineering,1512,251,0.109443,893.915673,0.166005
2,Finance,570,55,0.141591,509.237240,0.096491
5,Manufacturing,1808,162,0.127020,200.211863,0.089602
3,Human Resources,503,23,0.132382,212.396631,0.045726
7,Supply Chain,904,31,0.108829,-142.554693,0.034292
6,Sales,739,18,0.114442,-179.332614,0.024357
0,Customer Support,535,8,0.158424,-284.838646,0.014953


## 6. Validation

Every row must be `PASS`, including policy freeze, one-time test access, budget enforcement, employee-disjoint calibration, scenario coverage, and the human-review restriction.

In [8]:
validation

,check,status,observed,requirement,details
0,Configured base model remains selected,PASS,Logistic Regression,Logistic Regression,Policy evaluation must reuse the selected model.
1,Configured calibration method remains selected,PASS,Sigmoid,Sigmoid,Policy evaluation must reuse the approved prob...
2,Policy was frozen before test-target access,PASS,hash=96a85ef6dff7...; test access during selec...,Frozen hash exists; no test access during sele...,Test outcomes cannot influence the policy defi...
3,Test target is accessed exactly once,PASS,1,1,The final test is a once-only evaluation.
4,Final test snapshot is exact,PASS,['2025-06-30'],['2025-06-30'],Only the reserved chronological period is eval...
5,Final test population is sufficiently large,PASS,6641 rows; 803 positives,>= 1000 rows; >= 100 positives,Final metrics require a useful future population.
6,Final test probabilities are finite and bounded,PASS,min=0.011022; max=0.396046,"All finite and inside [0, 1]",The final calibrated model must output valid p...
7,Every candidate policy is compared in both per...,PASS,"{'2024 validation': 6, '2025 final test': 6}",6 per period,"Fixed threshold, top-k, and cost policies cann..."
8,Old fixed 0.50 threshold remains a comparator,PASS,"['budget_expected_value', 'cost_optimal_thresh...",fixed_threshold included,The former no-op rule is retained for honest c...
9,Selected policy is the constrained expected-va...,PASS,budget_expected_value,budget_expected_value,The choice follows the committed capacity ques...
